# Practice 13 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [2]:
import pandas as pd
import numpy as np

---
## Level 1 — Basics

### Task 1: MultiIndex
A store inventory DataFrame has been created for you.

1. Set a MultiIndex from `store` and `category`, then sort it
2. Retrieve all rows for `'Store B'`
3. Retrieve the single row for `('Store A', 'Electronics')`
4. Find mean price per store using NumPy

In [ ]:
# Starter data — don't change this
np.random.seed(5)
stores     = ['Store A', 'Store B', 'Store C']
categories = ['Electronics', 'Clothing', 'Food']
inventory = pd.DataFrame({
    'store':    stores * 3,
    'category': [c for c in categories for _ in stores],
    'price':    np.round(np.random.uniform(5, 500, size=9), 2),
    'stock':    np.random.randint(10, 200, size=9),
})

# Your code here
inventory = inventory.set_index(['store','category']).sort_index()
inventory.loc['Store B']
inventory.loc[('Store A','Electronics')]
for i in inventory.index.get_level_values('store').unique():
    p = inventory.loc[i,'price']
    print(i, np.mean(p))

#inventory.loc[(slice(None), 'Food'),:]

Store A 319.5733333333333
Store B 314.7966666666667
Store C 189.01999999999998


,,price,stock
store,category,,
Store A,Food,384.12,75
Store B,Food,261.62,185
Store C,Food,151.92,40


### Task 2: Duplicates
A subscriptions DataFrame has been created for you.

1. Count fully duplicate rows
2. Drop duplicates, keeping the **last** occurrence
3. Find which `user_id` values appear more than once
4. Show transaction counts per `plan`, sorted highest first

In [27]:
# Starter data — don't change this
subs = pd.DataFrame({
    'user_id': ['U1', 'U2', 'U3', 'U2', 'U4', 'U5', 'U5', 'U1', 'U3', 'U4'],
    'plan':    ['Basic', 'Pro', 'Basic', 'Pro', 'Enterprise',
                'Basic', 'Basic', 'Basic', 'Basic', 'Enterprise'],
    'amount':  [9.99, 29.99, 9.99, 29.99, 99.99, 9.99, 9.99, 9.99, 9.99, 99.99],
    'month':   ['Jan', 'Jan', 'Jan', 'Feb', 'Jan', 'Feb', 'Feb', 'Feb', 'Mar', 'Feb'],
})

# Your code here
sum(subs.duplicated())
subs = subs.drop_duplicates(keep='last')
subs['user_id'].value_counts()[subs['user_id'].value_counts()>1].index
subs['plan'].value_counts().sort_values(ascending=False)

plan
Basic         5
Pro           2
Enterprise    2
Name: count, dtype: int64

---
## Level 2 — Transformations

### Task 3: stack() and unstack() ✨ New

`stack()` folds column labels **into** the row index, converting wide format → long format.
The result is a Series (or DataFrame) with a MultiIndex on the rows.

`unstack()` is the inverse — it pulls an inner row-index level **out** into columns.
You can specify which level to unstack with `unstack(level=...)` (default: innermost).

```python
df.stack()           # columns → innermost row index level
df.unstack()         # innermost row level → columns
df.unstack(level=0)  # outermost row level → columns
```

A wide DataFrame of seasonal temperatures by city has been created for you  
(cities as the index, months as columns).

1. Use `.stack()` to reshape it so each row is one `(city, month)` pair with a single temperature value. Store the result as `temps_long`.
2. From `temps_long`, use `.unstack()` to restore the original wide shape.
3. From `temps_long`, use `.unstack(level=0)` — what do the columns now represent?
4. Find the city with the highest mean temperature across all months using NumPy

In [4]:
# Starter data — don't change this
temps = pd.DataFrame({
    'city': ['New York', 'Chicago', 'Miami', 'Denver', 'Seattle'],
    'Jan':  [32, 22, 68, 30, 42],
    'Apr':  [54, 50, 78, 52, 52],
    'Jul':  [82, 78, 89, 76, 67],
    'Oct':  [59, 54, 79, 54, 52],
}).set_index('city')

# Your code here
temps_long = temps.stack()
temps_long.unstack()
temps_long.unstack(level=0) ## now columns are cities 
temps_long

mean_temp = {}

for city in temps_long.index.get_level_values('city').unique():
    mean_temp[city] = np.mean(temps_long.loc[city])
    print(city, np.mean(temps_long.loc[city]))
max(mean_temp, key=mean_temp.get)

temps.mean(axis=1).idxmax()

temps.index.get_level_values('city')

New York 56.75
Chicago 51.0
Miami 78.5
Denver 53.0
Seattle 53.25


Index(['New York', 'Chicago', 'Miami', 'Denver', 'Seattle'], dtype='object', name='city')

### Task 4: Custom Column Logic
A student DataFrame has been created for you.

1. Add a `grade` column: `'A'` if `score` ≥ 85, `'B'` if ≥ 70, else `'C'`
2. Add a `scholarship` column: `True` if grade is `'A'` **and** `attendance` ≥ 90
3. Add a `log_score` column using NumPy
4. Count students per grade

In [78]:
# Starter data — don't change this
np.random.seed(12)
students = pd.DataFrame({
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace', 'Hank'],
    'score':      np.random.randint(55, 100, size=8),
    'attendance': np.random.randint(70, 100, size=8),
    'subject':    np.random.choice(['Math', 'Science', 'English'], size=8),
})

# Your code here
students['grade']=pd.cut(students['score'], bins=[-np.inf,70,85,np.inf],
                         labels=['C','B','A'],right=False)
students['scholarship'] = (students['grade']=='A') & (students['attendance']>=90)
students['log_score'] = np.log(students['score'])
students['grade'].value_counts()


grade
C    6
B    2
A    0
Name: count, dtype: int64

---
## Level 3 — Aggregation

### Task 5: .xs() — Cross-Section ✨ New

`.xs()` selects a cross-section of a MultiIndex DataFrame by label at a **specific level**.
It's especially useful for selecting by a *non-outermost* level — something `.loc[]` can't do as cleanly.

```python
df.xs('Q2', level='quarter')   # all rows where quarter == 'Q2' (any region)
df.xs('West', level='region')  # all rows where region == 'West' (any quarter)
```

A quarterly regional sales DataFrame with a `(region, quarter)` MultiIndex has been created for you.

1. Use `.xs()` to retrieve all rows for `'Q3'` (across all regions)
2. Use `.xs()` to retrieve all rows for the `'West'` region
3. Find which region had the highest total revenue in Q3 using NumPy
4. Find mean revenue per quarter — group by the `'quarter'` level

In [79]:
# Starter data — don't change this
np.random.seed(8)
regions  = ['North', 'East', 'South', 'West']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']
sales_q = pd.DataFrame({
    'region':  regions * 4,
    'quarter': [q for q in quarters for _ in regions],
    'revenue': np.random.randint(40_000, 200_000, size=16),
    'units':   np.random.randint(50, 500, size=16),
}).set_index(['region', 'quarter']).sort_index()

# Your code here
sales_q.xs('Q3', level = 'quarter')
sales_q.xs('West', level='region')

q3 = sales_q.xs('Q3', level = 'quarter')
q3 = sales_q.xs('Q3', level='quarter').assign(total=lambda d: d['revenue'] * d['units'])
#q3['total'] = q3['revenue']*q3['units']
q3['total'].idxmax()

sales_q.groupby('quarter')['revenue'].mean()

quarter
Q1    116984.50
Q2    124518.25
Q3    109922.25
Q4    170671.25
Name: revenue, dtype: float64

### Task 6: Rolling Windows
A daily sales DataFrame has been created for you.

1. Compute a 5-day rolling mean and rolling std of `sales`
2. Add an `above_avg` column — `True` when `sales` exceeds its 5-day rolling mean
3. Count `True` values in `above_avg`
4. Find the date where the gap between `sales` and its 5-day rolling mean was largest

In [80]:
# Starter data — don't change this
np.random.seed(99)
daily = pd.DataFrame({
    'date':  pd.date_range('2024-06-01', periods=60, freq='B'),
    'sales': np.round(np.random.uniform(500, 5000, size=60), 2),
}).set_index('date')

# Your code here
daily.rolling(5).agg(['mean','std'])
daily['r5m'] = daily.rolling(5).agg('mean')
daily['above_avg'] = daily['sales']>daily['r5m']
daily['above_avg'].sum()

(daily['sales'] -daily['r5m']).idxmax()


Timestamp('2024-07-23 00:00:00')

---
## Level 4 — Real-world

### Task 7: Full Pipeline
Load the tips dataset and run a full analysis:

1. Add a `tip_pct` column = `tip / total_bill`
2. Bin `size` into 3 groups labelled `['Small', 'Medium', 'Large']`, store as `size_group`
3. Compute mean `tip_pct` grouped by `day` and `time`
4. Use `.unstack()` on the result so days are rows and meal times are columns
5. Use `.xs()` to pull out just the `'Dinner'` column from the grouped result (before unstacking)
6. Find the correlation between `total_bill` and `tip` using NumPy

In [76]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv'
tips = pd.read_csv(url)

# Your code here
tips['tip_pct'] = tips['tip']/tips['total_bill']
tips['size_group'] = pd.cut(tips['size'],3,labels=['Small','Medium','Large'])
g = tips.groupby(['day','time'])['tip_pct'].agg('mean')
g.unstack()
g
g.xs('Dinner',level='time')
np.corrcoef(tips['total_bill'],tips['tip'])

array([[1.        , 0.67573411],
       [0.67573411, 1.        ]])